# Emergency Action Recognition: R3D-18 Training on UR Fall Detection (URFD)

This Google Colab notebook provides the reproducible training and evaluation pipeline for **Emergency Action Recognition (NORMAL vs. FALL)** on a Tesla T4 GPU.

### Pipeline Overview
- **Model**: 3D ResNet-18 (`R3D-18`)
- **Initialization**: Official Torchvision Kinetics-400 Pre-trained Weights (`R3D_18_Weights.KINETICS400_V1`)
- **Dataset**: UR Fall Detection (URFD) from University of Rzeszow (30 Fall + 40 Normal ADL sequences)
- **Training**: Two-Stage Transfer Learning (Stage 1: Head Warm-up, Stage 2: Differential Fine-tuning of layers 3 & 4)
- **Storage**: Persistent Google Drive dataset, checkpoint, and experiment logging

In [ ]:
# CELL 1 — Google Drive Persistent Storage Configuration
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configure persistent Drive paths
DRIVE_ROOT = "/content/drive/MyDrive/emergency-vision-ai"
DRIVE_DATASET_DIR = os.path.join(DRIVE_ROOT, "data", "urfd")
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "models", "action_recognition")
DRIVE_RESULTS_DIR = os.path.join(DRIVE_ROOT, "results", "urfd_r3d18")

# 3. Create persistent directories
for path in [DRIVE_DATASET_DIR, DRIVE_CHECKPOINT_DIR, DRIVE_RESULTS_DIR]:
    os.makedirs(path, exist_ok=True)

print("Persistent Google Drive paths configured:")
print(f"  DRIVE_ROOT:           {DRIVE_ROOT}")
print(f"  DRIVE_DATASET_DIR:    {DRIVE_DATASET_DIR}")
print(f"  DRIVE_CHECKPOINT_DIR: {DRIVE_CHECKPOINT_DIR}")
print(f"  DRIVE_RESULTS_DIR:    {DRIVE_RESULTS_DIR}")

In [ ]:
# CELL 2 — Clone / Update Repository
import os

REPO_URL = "https://github.com/mukhammadiev01-1/emergency-vision-ai.git"
LOCAL_REPO_DIR = "/content/emergency-vision-ai"

if not os.path.exists(LOCAL_REPO_DIR):
    print(f"Cloning {REPO_URL} into {LOCAL_REPO_DIR}...")
    !git clone {REPO_URL} {LOCAL_REPO_DIR}
else:
    print(f"Repository exists at {LOCAL_REPO_DIR}. Pulling latest updates...")
    %cd {LOCAL_REPO_DIR}
    !git pull origin main

%cd {LOCAL_REPO_DIR}
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# CELL 3 — Install Dependencies
import sys

print("Installing worker and ML dependencies...")
!pip install -q --upgrade pip
!pip install -q -r requirements-worker.txt
!pip install -q certifi torchvision av opencv-python-headless

# Ensure repo root is in Python sys.path
if LOCAL_REPO_DIR not in sys.path:
    sys.path.insert(0, LOCAL_REPO_DIR)

print("Dependencies installed successfully.")

In [ ]:
# CELL 4 — Verify GPU
import torch

print("=" * 50)
print("CUDA / GPU ACCELERATION CHECK")
print("=" * 50)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available in this Colab runtime!\n"
        "Please go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU and restart."
    )

print(f"CUDA Version:     {torch.version.cuda}")
print(f"Device Count:     {torch.cuda.device_count()}")
print(f"GPU Name:         {torch.cuda.get_device_name(0)}")
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"Total GPU Memory: {gpu_mem_gb:.2f} GB")
print("=" * 50)

In [ ]:
# CELL 5 — Link Dataset from Google Drive
import os
import shutil

LOCAL_DATA_DIR = os.path.join(LOCAL_REPO_DIR, "data", "urfd")
os.makedirs(os.path.dirname(LOCAL_DATA_DIR), exist_ok=True)

# Check if dataset exists in Drive
drive_fall_dir = os.path.join(DRIVE_DATASET_DIR, "videos", "fall")
drive_normal_dir = os.path.join(DRIVE_DATASET_DIR, "videos", "normal")

fall_count = len([f for f in os.listdir(drive_fall_dir) if f.endswith('.mp4')]) if os.path.exists(drive_fall_dir) else 0
normal_count = len([f for f in os.listdir(drive_normal_dir) if f.endswith('.mp4')]) if os.path.exists(drive_normal_dir) else 0

if fall_count < 30 or normal_count < 40:
    print(f"Dataset incomplete in Drive ({fall_count} falls, {normal_count} normals). Downloading to Drive...")
    !python scripts/download_urfd.py --output-dir "{DRIVE_DATASET_DIR}" --format mp4
else:
    print(f"Verified existing dataset in Google Drive: {fall_count} falls, {normal_count} normal ADLs.")

# Create symlink from local repository data path to Drive dataset
if os.path.islink(LOCAL_DATA_DIR):
    os.unlink(LOCAL_DATA_DIR)
elif os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)

os.symlink(DRIVE_DATASET_DIR, LOCAL_DATA_DIR)
print(f"Symlinked: {LOCAL_DATA_DIR} -> {DRIVE_DATASET_DIR}")

In [ ]:
# CELL 6 — Verify Dataset
import os

fall_videos = sorted([f for f in os.listdir(os.path.join(LOCAL_DATA_DIR, "videos", "fall")) if f.endswith('.mp4')])
normal_videos = sorted([f for f in os.listdir(os.path.join(LOCAL_DATA_DIR, "videos", "normal")) if f.endswith('.mp4')])

total_size_bytes = 0
for root, _, files in os.walk(LOCAL_DATA_DIR):
    for f in files:
        total_size_bytes += os.path.getsize(os.path.join(root, f))

print("=" * 50)
print("URFD DATASET VERIFICATION")
print("=" * 50)
print(f"FALL Videos Count:   {len(fall_videos)} / 30")
print(f"NORMAL Videos Count: {len(normal_videos)} / 40")
print(f"Total Videos Count:  {len(fall_videos) + len(normal_videos)} / 70")
print(f"Total Dataset Size:  {total_size_bytes / (1024 * 1024):.2f} MB")
print(f"Sample FALL Files:   {fall_videos[:3]} ... {fall_videos[-1:]}")
print(f"Sample NORMAL Files: {normal_videos[:3]} ... {normal_videos[-1:]}")
print("=" * 50)

if len(fall_videos) != 30 or len(normal_videos) != 40:
    raise ValueError(f"Expected 30 fall and 40 normal sequences, found {len(fall_videos)} and {len(normal_videos)}.")
print("Dataset verification passed.")

In [ ]:
# CELL 7 — Verify Deterministic Splits
from apps.worker.app.datasets.urfd_dataset import create_urfd_splits, LABEL_FALL, LABEL_NORMAL

train_ds, val_ds, test_ds = create_urfd_splits(LOCAL_DATA_DIR, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42)

def count_split_classes(ds):
    falls = sum(1 for s in ds.samples if s.label == LABEL_FALL)
    normals = sum(1 for s in ds.samples if s.label == LABEL_NORMAL)
    return falls, normals

tr_f, tr_n = count_split_classes(train_ds)
val_f, val_n = count_split_classes(val_ds)
te_f, te_n = count_split_classes(test_ds)

print("=" * 60)
print("DETERMINISTIC SPLIT VERIFICATION (Seed=42)")
print("=" * 60)
print(f"{'Split':<12} | {'Total':<8} | {'FALL (1)':<10} | {'NORMAL (0)':<10}")
print("-" * 50)
print(f"{'Train':<12} | {len(train_ds):<8} | {tr_f:<10} | {tr_n:<10}")
print(f"{'Validation':<12} | {len(val_ds):<8} | {val_f:<10} | {val_n:<10}")
print(f"{'Test':<12} | {len(test_ds):<8} | {te_f:<10} | {te_n:<10}")
print("-" * 50)
print(f"{'Total':<12} | {len(train_ds) + len(val_ds) + len(test_ds):<8} | {tr_f + val_f + te_f:<10} | {tr_n + val_n + te_n:<10}")
print("=" * 60)

# Verify sequence isolation (zero data leakage)
train_seqs = {s.sequence_id for s in train_ds.samples}
val_seqs = {s.sequence_id for s in val_ds.samples}
test_seqs = {s.sequence_id for s in test_ds.samples}

assert train_seqs.isdisjoint(val_seqs), "Error: Overlap between train and val sequences!"
assert train_seqs.isdisjoint(test_seqs), "Error: Overlap between train and test sequences!"
assert val_seqs.isdisjoint(test_seqs), "Error: Overlap between val and test sequences!"
print("Strict sequence isolation verified: No sequence overlap across splits.")

In [ ]:
# CELL 8 — Verify Model Initialization
import torch
import torch.nn as nn
from scripts.train_action_model import build_r3d18_model

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_r3d18_model(num_classes=2, pretrained=True, device=device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("R3D-18 MODEL ARCHITECTURE SUMMARY")
print("=" * 60)
print(f"Backbone:               ResNet3D-18 (R3D-18)")
print(f"Pretrained Weights:     Torchvision Kinetics-400 (DEFAULT)")
print(f"Classifier Head:        {model.fc}")
print(f"Total Parameters:       {total_params:,}")
print(f"Trainable Parameters:   {trainable_params:,}")
print(f"Device:                 {next(model.parameters()).device}")
print("=" * 60)

In [ ]:
# CELL 9 — DataLoader Smoke Test
from torch.utils.data import DataLoader

BATCH_SIZE = 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Sample single batch
videos, labels = next(iter(train_loader))

print("=" * 50)
print("DATALOADER BATCH SMOKE TEST")
print("=" * 50)
print(f"Video Tensor Shape:  {videos.shape} (Expected: [B, 3, 16, 112, 112])")
print(f"Labels Tensor:       {labels.tolist()}")
print(f"Video Tensor Dtype:  {videos.dtype}")
print(f"Batch Size:          {videos.size(0)}")
print("=" * 50)

assert videos.shape == torch.Size([BATCH_SIZE, 3, 16, 112, 112]), f"Unexpected shape: {videos.shape}"
assert labels.size(0) == BATCH_SIZE
print("DataLoader smoke test passed.")

In [ ]:
# CELL 10 — GPU Forward-Pass Benchmark
import time

model.eval()
dummy_input = torch.randn(BATCH_SIZE, 3, 16, 112, 112, device=device)

# 1. Warm-up pass
torch.cuda.synchronize()
t_warmup_start = time.perf_counter()
with torch.no_grad():
    _ = model(dummy_input)
torch.cuda.synchronize()
warmup_time_ms = (time.perf_counter() - t_warmup_start) * 1000.0

# 2. Timed benchmark passes
NUM_ITERATIONS = 50
torch.cuda.synchronize()
t_start = time.perf_counter()
with torch.no_grad():
    for _ in range(NUM_ITERATIONS):
        _ = model(dummy_input)
torch.cuda.synchronize()
total_time = time.perf_counter() - t_start

avg_latency_ms = (total_time / NUM_ITERATIONS) * 1000.0
throughput_clips_per_sec = (NUM_ITERATIONS * BATCH_SIZE) / total_time

print("=" * 50)
print("GPU FORWARD-PASS BENCHMARK (Tesla T4)")
print("=" * 50)
print(f"Warm-up Latency:     {warmup_time_ms:.2f} ms")
print(f"Avg Latency / Batch: {avg_latency_ms:.2f} ms (Batch Size = {BATCH_SIZE})")
print(f"Latency / Clip:      {avg_latency_ms / BATCH_SIZE:.2f} ms")
print(f"Throughput:          {throughput_clips_per_sec:.2f} clips/sec")
print("=" * 50)

In [ ]:
# CELL 11 — Full Action Model Training
import os

CHECKPOINT_FILENAME = "r3d18_urfd_best.pth"

print("=" * 60)
print("LAUNCHING FULL URFD R3D-18 ACTION MODEL TRAINING")
print("=" * 60)

!python scripts/train_action_model.py --dataset-root "{LOCAL_DATA_DIR}" --output-dir "{DRIVE_CHECKPOINT_DIR}" --checkpoint-name "{CHECKPOINT_FILENAME}" --stage1-epochs 5 --stage2-epochs 20 --batch-size 4 --device cuda --seed 42

In [ ]:
# CELL 12 — Verify Saved Checkpoint in Google Drive
import os
import time

saved_checkpoint_path = os.path.join(DRIVE_CHECKPOINT_DIR, "r3d18_urfd_best.pth")

if not os.path.exists(saved_checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found at persistent path: {saved_checkpoint_path}")

stat = os.stat(saved_checkpoint_path)
size_mb = stat.st_size / (1024 * 1024)
mod_time = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(stat.st_mtime))

print("=" * 50)
print("CHECKPOINT VERIFICATION (Google Drive)")
print("=" * 50)
print(f"Location:      {saved_checkpoint_path}")
print(f"File Size:     {size_mb:.2f} MB")
print(f"Modified Time: {mod_time}")
print("=" * 50)

In [ ]:
# CELL 13 — Model Evaluation on Test Split
print("=" * 60)
print("EVALUATING BEST MODEL CHECKPOINT ON TEST SPLIT")
print("=" * 60)

!python scripts/evaluate_action_model.py --checkpoint "{DRIVE_CHECKPOINT_DIR}/r3d18_urfd_best.pth" --dataset-root "{LOCAL_DATA_DIR}" --batch-size 4 --device cuda --seed 42

In [ ]:
# CELL 14 — Save Results and Artifacts to Google Drive
import json
import numpy as np
from scripts.evaluate_action_model import evaluate_checkpoint

eval_metrics = evaluate_checkpoint(
    checkpoint_path=os.path.join(DRIVE_CHECKPOINT_DIR, "r3d18_urfd_best.pth"),
    dataset_root=LOCAL_DATA_DIR,
    device_str="cuda",
    seed=42,
)

# 1. Save evaluation JSON
serializable_eval = {
    k: (v.tolist() if isinstance(v, np.ndarray) else float(v))
    for k, v in eval_metrics.items()
}
eval_json_path = os.path.join(DRIVE_RESULTS_DIR, "evaluation_metrics.json")
with open(eval_json_path, "w") as f:
    json.dump(serializable_eval, f, indent=2)

# 2. Save confusion matrix CSV
cm_csv_path = os.path.join(DRIVE_RESULTS_DIR, "confusion_matrix.csv")
cm = eval_metrics["confusion_matrix"]
with open(cm_csv_path, "w") as f:
    f.write("Actual/Predicted,Predicted_NORMAL,Predicted_FALL\n")
    f.write(f"Actual_NORMAL,{cm[0, 0]},{cm[0, 1]}\n")
    f.write(f"Actual_FALL,{cm[1, 0]},{cm[1, 1]}\n")

# 3. Save experiment configuration
config = {
    "model": "ResNet3D-18 (R3D-18)",
    "initialization": "Torchvision Kinetics-400 (DEFAULT)",
    "dataset": "UR Fall Detection (URFD)",
    "stage1_epochs": 5,
    "stage2_epochs": 20,
    "batch_size": 4,
    "clip_length": 16,
    "spatial_size": [112, 112],
    "optimizer": "AdamW",
    "loss": "Weighted CrossEntropyLoss",
    "device": "NVIDIA Tesla T4 (CUDA)",
    "checkpoint": os.path.join(DRIVE_CHECKPOINT_DIR, "r3d18_urfd_best.pth"),
}
config_json_path = os.path.join(DRIVE_RESULTS_DIR, "experiment_config.json")
with open(config_json_path, "w") as f:
    json.dump(config, f, indent=2)

print("Artifacts saved to Google Drive:")
print(f"  - {eval_json_path}")
print(f"  - {cm_csv_path}")
print(f"  - {config_json_path}")

In [ ]:
# CELL 15 — Final ML Experiment Summary
print("\n" + "=" * 65)
print("         EMERGENCY ACTION RECOGNITION: FINAL EXPERIMENT SUMMARY")
print("=" * 65)
print(f"Dataset:                UR Fall Detection (URFD) — 70 sequences")
print(f"Splits:                 Train (49) / Validation (10) / Test (11)")
print(f"Model Architecture:     ResNet3D-18 (R3D-18)")
print(f"Backbone Weights:       Official Torchvision Kinetics-400 Pretrained")
print(f"Training Strategy:      Stage 1 (5 Ep Head) + Stage 2 (20 Ep Layer3/4)")
print(f"Batch Size & Frames:    Batch = 4 | Frames = 16 | Resolution = 112x112")
print(f"Hardware Accelerator:   {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Best Checkpoint:        {os.path.join(DRIVE_CHECKPOINT_DIR, 'r3d18_urfd_best.pth')}")
print("-" * 65)
print(f"Test Accuracy:          {eval_metrics['accuracy'] * 100:.2f}%")
print(f"FALL Precision / Recall:{eval_metrics['fall_precision'] * 100:.2f}% / {eval_metrics['fall_recall'] * 100:.2f}%")
print(f"FALL F1-Score:          {eval_metrics['fall_f1'] * 100:.2f}%")
print(f"NORMAL F1-Score:        {eval_metrics['normal_f1'] * 100:.2f}%")
print(f"Macro F1-Score:         {eval_metrics['macro_f1'] * 100:.2f}%")
print("=" * 65)